# 🛠️ Notebook 2: Stack Overflow — Implementation

## 🛠️ Setup

```bash
cd 07-object-oriented-design/stack-overflow
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


In [ ]:
from dataclasses import dataclass, field
from typing import Optional, List
import itertools

REP_UPVOTE_QUESTION = 5
REP_UPVOTE_ANSWER   = 10
REP_DOWNVOTE        = -2
REP_ACCEPTED_BONUS  = 15

@dataclass
class User:
    id: str
    name: str
    reputation: int = 0

_pid = itertools.count(1)

class Post:
    def __init__(self, author: User, body: str):
        self.id = next(_pid)
        self.author = author
        self.body = body
        self.votes = {}     # user_id -> +1/-1
        self.comments = []
    @property
    def score(self):
        return sum(self.votes.values())
    def comment(self, user, text):
        self.comments.append((user.id, text))

class Question(Post):
    def __init__(self, author, title, body, tags):
        super().__init__(author, body)
        self.title = title
        self.tags = list(tags)
        self.answers: List['Answer'] = []
        self.accepted: Optional['Answer'] = None
    def add_answer(self, ans): self.answers.append(ans)
    def accept(self, ans):
        if ans not in self.answers: raise ValueError('not an answer to this question')
        self.accepted = ans
        ans.author.reputation += REP_ACCEPTED_BONUS

class Answer(Post):
    def __init__(self, author, question: Question, body):
        super().__init__(author, body)
        self.question = question
        question.add_answer(self)

def vote(post: Post, voter: User, direction: int):
    assert direction in (1, -1)
    if voter.id == post.author.id:
        raise ValueError('cannot vote on your own post')
    prev = post.votes.get(voter.id, 0)
    post.votes[voter.id] = direction
    delta = direction - prev
    # Reputation accrues to the post's author.
    per_upvote = REP_UPVOTE_ANSWER if isinstance(post, Answer) else REP_UPVOTE_QUESTION
    if delta > 0:
        post.author.reputation += per_upvote * delta
    elif delta < 0:
        post.author.reputation += REP_DOWNVOTE * (-delta)


## Walk-through

In [ ]:
ada   = User('u1','Ada')
grace = User('u2','Grace')
bob   = User('u3','Bob')

q = Question(ada, 'What is OOD?', 'I want to learn OOD.', ['ood','design'])
a1 = Answer(grace, q, 'Start with SOLID principles ...')
a2 = Answer(bob, q, 'Practice with classic problems like parking lot.')

vote(q,  grace, +1)   # Ada gets +5
vote(a1, ada,   +1)   # Grace gets +10
vote(a1, bob,   +1)   # Grace gets +10
vote(a2, ada,   -1)   # Bob gets -2

q.accept(a1)          # Grace gets +15 bonus

for u in (ada, grace, bob):
    print(f'{u.name}: rep={u.reputation}')

print('scores  q={} a1={} a2={}'.format(q.score, a1.score, a2.score))
print('accepted answer id:', q.accepted.id)


### Rules exercised
- Self-voting is blocked.
- Changing your vote (up→down) adjusts reputation correctly.
- Accepting an answer is a one-way door in this simple model.

### Extensions
- Tags index for search.
- Badges based on reputation milestones.
- Edit history / revisions on posts.
- Soft-delete (flagged / closed) questions.